IMPORT

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

imprt the data

In [ ]:
df =  pd.read_csv('/content/drive/MyDrive/vegindex/BEJAIA.csv')
pd.set_option('display.max_columns', None)
print(df.head())
print(df.columns)

In [ ]:
columns_to_process = ['sand % topsoil', 'silt % topsoil', 'clay % topsoil', 'pH water topsoil',
                      'OC % topsoil', 'N % topsoil', 'BS % topsoil', 'CEC topsoil',
                      'CEC clay topsoil', 'CaCO3 % topsoil', 'BD topsoil', 'C/N topsoil']

# Remplacement des virgules par des points et conversion en float
for col in columns_to_process:
    df[col] = df[col].str.replace(',', '.').astype(float)

print(df.head())

In [ ]:
#df.select_dtypes('object').columns
df['LCCCODE'].value_counts()

In [ ]:
code = {'21450':1,'0004 // 0003':2,'0003 / 0004':3,'21446 // 21450-121340 / 21454':4,'21518':5
        ,'21497-121340':6,'21499-121340':7,'11498':8,'21454 // 21446 // 21450':9
        ,'20049 // 20058':10,'7001 // 8001':11,'0010':12,'21496-121340 // 21497-129401':13,'21497-15045':13,'0011':14}
df['LCCCODE'] = df['LCCCODE'].map(code)
print(df.head(10))

In [8]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['fire'] = le.fit_transform(df['fire'])
print(df.head(100))

     fid  GRIDCODE          AREA  LCCCODE  sand % topsoil  silt % topsoil  \
0   1020       210  5.688647e+07       11            55.9           20.24   
1   1020       210  5.688647e+07       11            55.9           20.24   
2   1020       210  5.688647e+07       11            55.9           20.24   
3   1355        41  2.289760e+05       13            55.9           20.24   
4   1391        41  2.289841e+05       13            55.9           20.24   
..   ...       ...           ...      ...             ...             ...   
95  1394       130  3.041149e+08        1            55.9           20.24   
96  1394       130  3.041149e+08        1            55.9           20.24   
97  1394       130  3.041149e+08        1            55.9           20.24   
98  1394       130  3.041149e+08        1            55.9           20.24   
99  1394       130  3.041149e+08        1            55.9           20.24   

    clay % topsoil  pH water topsoil  OC % topsoil  N % topsoil  BS % topso

In [6]:
y=df['fire']
print(y.value_counts)

<bound method IndexOpsMixin.value_counts of 0          1
1          1
2          1
3          1
4          1
          ..
2549113    1
2549114    1
2549115    1
2549116    1
2549117    1
Name: fire, Length: 2549118, dtype: int64>


In [ ]:
df['DATE'] = pd.to_datetime(df['DATE'])
df['Day'] = df['DATE'].dt.day
df['Month'] = df['DATE'].dt.month_name()
df['Year'] = df['DATE'].dt.year

In [ ]:
df.dtypes

In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix
import seaborn as sns
from scipy.stats import mode
from sklearn.tree import DecisionTreeClassifier

y_test_combined_list = []
predictions = []

fire_indices = np.where(y == 0)[0]

# Définir le nombre d'instances de la classe Fire à inclure à chaque itération
fire_samples = len(fire_indices)

# Initialisation de StratifiedShuffleSplit en spécifiant le nombre de splits
stratified_shuffle_split = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

# Utilisation de l'objet StratifiedShuffleSplit pour obtenir les index des ensembles d'entraînement et de validation
for train_index, test_index in stratified_shuffle_split.split(X_train, y_train):
    # Sélectionner les instances de la classe Fire pour l'entraînement
    X_train_fire = X_train.iloc[fire_indices]

    # Sélectionner une partie aléatoire de la classe Not Fire pour l'entraînement
    X_train_not_fire = X_train[train_index[y_train[train_index] == 1]][:fire_samples]

    # Combiner les données d'entraînement de Fire et Not Fire
    X_train_combined = np.concatenate((X_train_fire, X_train_not_fire), axis=0)

    # Les étiquettes correspondantes
    y_train_combined = np.concatenate(([0] * fire_samples, [1] * fire_samples))


    # Utiliser X_train_combined, y_train_combined pour entraîner votre modèle
    # Utiliser X.iloc[test_index], y.iloc[test_index] pour valider votre modèle


In [22]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Initialisation de StratifiedShuffleSplit en spécifiant le nombre de splits
stratified_shuffle_split = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

for train_index, test_index in stratified_shuffle_split.split(X_train, y_train):
    # Sélectionner les instances de la classe Fire pour l'entraînement
    X_train_fire = X_train[y_train == 0]

    # Sélectionner une partie aléatoire de la classe Not Fire pour l'entraînement
    X_train_not_fire = X_train[(y_train == 1) & train_index][:len(X_train_fire)]

    # Combiner les données d'entraînement de Fire et Not Fire
    X_train_combined = pd.concat([X_train_fire, X_train_not_fire])

    # Les étiquettes correspondantes
    y_train_combined = pd.concat([y_train[y_train == 0], y_train[(y_train == 'Not Fire') & train_index][:len(X_train_fire)]])

    # Initialiser et entraîner le modèle RandomForestClassifier
    model = RandomForestClassifier()
    model.fit(X_train_combined, y_train_combined)

    # Prédiction sur l'ensemble de test
    y_pred = model.predict(X.iloc[test_index])

    # Calculer et imprimer la précision
    accuracy = accuracy_score(y.iloc[test_index], y_pred)
    print(f'Accuracy: {accuracy}')


In [ ]:
df_imputed = df.fillna(df.mean())

In [ ]:
sns.heatmap(df.isna())

In [ ]:
fire = df[df['fire'] == 'Fire']
notfire = df[df['fire'] == 'Not Fire']


plt.figure()
sns.distplot(fire['GRIDCODE'])
sns.distplot(notfire['GRIDCODE'])
plt.legend(['Fire', 'Not Fire'])
plt.xlabel('GRIDCODE')
plt.ylabel('Density')
plt.title(f'Distribution of GRIDCODE for Fire and Not Fire')
plt.show()


In [ ]:
sns.countplot(x='Data_x',hue='fire',data=df)

In [10]:
from sklearn.model_selection import train_test_split

trainset, testset = train_test_split(df, test_size=0.2, random_state=0)

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

X_train = trainset.drop(columns=['fire'])
y_train = trainset['fire']

print("X:")
print(X_train.head())
print("\ny:")
print(y_train.head())

from sklearn.model_selection import StratifiedShuffleSplit

X_test = testset.drop(columns=['fire'])
y_test = testset['fire']

print("X:")
print(X_train.head())
print("\ny:")
print(y_train.head())

Droping unuseffull columns

spliting

In [ ]:
import sklearn.model_selctionimport train_test_split
trainset ,testset = train_test_split(df, test_size=0.2, random_state=0)

estimier le nombre de vleur

In [ ]:
trainset['fire'].value_counts()
testset['fire'].value_counts()

l'encodage

In [ ]:
# variable de type object :

In [ ]:
def encodage(df):
  code = {'','',..}
  for col in df.select_dtypes('object').columns:
    df.loc[:,col] = df[col].map(code)

  return df

netoyage

In [ ]:
def imputation(df):
  return df.dropna(axis=0)

pretraitement

In [ ]:
def preprocessing(df):
  df = encodage(df)
  df = imputation(df)

  X = df.drop('fire',axis=1)
  Y